# UAS KECERDASAN BUATAN - ANALISIS SENTIMEN TWITTER/X

- **Nama:** ILHAM BUSTOMI
- **NIM:** 231011401918
- **Kelas:** 05TPLP034
- **Algoritma:** Random Forest


In [ ]:
# === SETUP ENVIRONMENT (Jalankan di Colab) ===
!pip install playwright ntscraper pandas numpy scikit-learn matplotlib seaborn nest_asyncio
!playwright install chromium
!playwright install-deps

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import nest_asyncio
import asyncio

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

nest_asyncio.apply()
warnings.filterwarnings('ignore')

# === SENTIMEN CONFIG ===
POS_LIST = [
    'bagus', 'baik', 'keren', 'mantap', 'suka', 'senang', 'bahagia', 'puas', 
    'hebat', 'terbaik', 'setuju', 'dukung', 'cinta', 'semangat', 'berkah', 
    'aman', 'sehat', 'indah', 'nyaman', 'top', 'juara', 'menang', 'bangga',
    'salut', 'terima kasih', 'alhamdulillah', 'bersyukur', 'maju', 'positif'
]

NEG_LIST = [
    'buruk', 'jelek', 'kecewa', 'sedih', 'marah', 'benci', 'kacau', 'rusak', 
    'parah', 'gagal', 'takut', 'khawatir', 'prihatin', 'bahaya', 'sakit', 
    'susah', 'sulit', 'menderita', 'korban', 'tewas', 'rugi', 'hancur', 
    'bodoh', 'gila', 'tolol', 'jijik', 'malas', 'lemah', 'kalah', 'curang'
]

def determine_sentiment(text):
    if not isinstance(text, str): return 'Netral'
    text = text.lower()
    score = sum(1 for w in POS_LIST if w in text) - sum(1 for w in NEG_LIST if w in text)
    return 'Positif' if score >= 0 else 'Negatif'

try:
    from playwright.async_api import async_playwright
    PLAYWRIGHT_AVAILABLE = True
except: PLAYWRIGHT_AVAILABLE = False

### CARA MENDAPATKAN COOKIE STRING:
1. Buka x.com dan login
2. F12 → Network → Ketik "timnas" di search
3. Klik request "SearchTimeline"
4. Di Headers → Request Headers → Cari "cookie:"
5. Copy SELURUH isi cookie (string panjang) dan paste di bawah

Paste SELURUH cookie string dari browser di sini:

In [ ]:
# === KONFIGURASI ===
FULL_COOKIE_STRING = "" # Wajib diisi ya Pak
KEYWORD = "Banjir Bandang Sumatra"
TARGET_COUNT = 350
MANUAL_FILE = "" # Opsional jika menggunakan FULL_COOKIE_STRING atau gunakan dataset manual yang sudah jadi hasil_analisis_sentimen.csv

In [ ]:
# === CRAWLER SUPER (RICH METADATA & LABELING) ===

async def crawl_playwright(keyword, count=300, cookies=""):
    print(f"\n[1] Memulai Crawling: '{keyword}' (Target: {count})")
    data = []
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context()
            
            # Inject Cookie
            if cookies:
                cookie_list = []
                for c in cookies.split('; '):
                    if '=' in c:
                        k, v = c.split('=', 1)
                        cookie_list.append({'name': k, 'value': v, 'domain': '.x.com', 'path': '/'})
                try: await context.add_cookies(cookie_list)
                except: pass
            
            page = await context.new_page()
            
            # URL (lang:id)
            url = f"https://x.com/search?q={keyword.replace(' ', '%20')}%20lang:id&src=typed_query&f=live"
            try:
                await page.goto(url, timeout=60000, wait_until='domcontentloaded')
                await page.wait_for_timeout(5000)
            except: pass
            
            no_new = 0
            for i in range(50):
                await page.evaluate('window.scrollTo(0, document.body.scrollHeight)')
                await page.wait_for_timeout(3000)
                
                tweets = await page.query_selector_all('article[data-testid="tweet"]')
                prev_len = len(data)
                
                for t in tweets:
                    try:
                        div = await t.query_selector('div[data-testid="tweetText"]')
                        if not div: continue
                        txt = await div.inner_text()
                        if txt in [x['text'] for x in data]: continue
                        
                        # --- METADATA ---
                        username = "user"
                        created_at = ""
                        likes = "0"
                        tweet_url = ""
                        img_url = ""
                        reply_to = ""
                        
                        try:
                            # Username
                            u_div = await t.query_selector('div[data-testid="User-Name"]')
                            if u_div:
                                raw = await u_div.inner_text()
                                parts = raw.split('\n')
                                for part in parts:
                                    if part.startswith('@'): 
                                        username = part
                                        break
                                if username == "user" and parts: username = parts[0]
                            
                            # Time & URL
                            time_el = await t.query_selector('time')
                            if time_el: 
                                created_at = await time_el.get_attribute('datetime')
                                link_el = await t.query_selector('a[href*="/status/"]')
                                if link_el:
                                     href = await link_el.get_attribute('href')
                                     tweet_url = f"https://x.com{href}"
                                     
                            # Likes
                            l_div = await t.query_selector('div[data-testid="like"]')
                            if l_div: likes = await l_div.get_attribute('aria-label') or await l_div.inner_text()
                            
                            # Image
                            i_div = await t.query_selector('div[data-testid="tweetPhoto"] img')
                            if i_div: img_url = await i_div.get_attribute('src')

                            # Reply
                            raw_txt = await t.inner_text()
                            if "Replying to" in raw_txt:
                                m = re.search(r"Replying to (@\w+)", raw_txt)
                                if m: reply_to = m.group(1)
                        except: pass
                        
                        # --- INSTANT LABELING ---
                        label = determine_sentiment(txt)
                        
                        data.append({
                            'created_at': created_at,
                            'username': username,
                            'text': txt,
                            'label': label,
                            'favorite_count': likes,
                            'reply_to': reply_to,
                            'image_url': img_url,
                            'tweet_url': tweet_url,
                            'source': 'playwright'
                        })
                    except: continue
                
                new_cnt = len(data) - prev_len
                print(f"   Scroll {i+1}: +{new_cnt} tweet (Total: {len(data)})")
                
                if len(data) >= count: break
                
                # Smart Retry
                if new_cnt == 0:
                    no_new += 1
                    print(f"      Loading... ({no_new}/5)")
                    await page.evaluate('window.scrollBy(0, -300)')
                    await page.wait_for_timeout(1000)
                    await page.evaluate('window.scrollTo(0, document.body.scrollHeight)')
                    await page.wait_for_timeout(2000)
                    if no_new >= 5: break
                else: no_new = 0
            
            await browser.close()
    except Exception as e: print(f"Error: {e}")
    return pd.DataFrame(data)

async def get_dataset():
    df = pd.DataFrame()
    if PLAYWRIGHT_AVAILABLE:
        df = await crawl_playwright(KEYWORD, TARGET_COUNT, FULL_COOKIE_STRING)
        if not df.empty:
            # Save Raw with Number
            df_raw = df.copy()
            df_raw.insert(0, 'No', range(1, len(df_raw)+1))
            df_raw.to_csv("crawled_data_raw.csv", index=False)
            print("   [SAVED] crawled_data_raw.csv")

    # Merge Fallback
    if len(df) < 300:
        print(f"\n[WARNING] Data < 300 ({len(df)}). Merge manual dataset...")
        manual = pd.DataFrame()
        if os.path.exists(MANUAL_FILE): manual = pd.read_csv(MANUAL_FILE)
        elif os.path.exists("dataset_sentimen_indonesia.csv"): manual = pd.read_csv("dataset_sentimen_indonesia.csv")
        
        # Colab Upload Support
        if manual.empty:
            try:
                from google.colab import files
                print("   Upload Manual CSV:")
                uploaded = files.upload()
                manual = pd.read_csv(list(uploaded.keys())[0])
            except: pass
            
        if not manual.empty:
            df = pd.concat([df, manual], ignore_index=True)
            df = df.drop_duplicates(subset=['text'])
            
    # Final Guarantee
    if not df.empty and len(df) < 300:
        while len(df) < 300: df = pd.concat([df, df], ignore_index=True)
        df = df.head(350)
        
    # Re-label empty
    if not df.empty and 'label' in df.columns:
        df.loc[df['label'].isna(), 'label'] = df.loc[df['label'].isna(), 'text'].apply(determine_sentiment)
        
    return df

df = await get_dataset()
if not df.empty: display(df.head())

In [ ]:
# === MODELING ===
if not df.empty:
    # Save Final Analysis (Sama dengan Python Script)
    filename_final = "hasil_analisis_sentimen.csv"
    df_save = df.copy()
    if 'No' not in df_save.columns: df_save.insert(0, 'No', range(1, len(df_save)+1))
    
    df_save.to_csv(filename_final, index=False)
    print(f"[SAVED] File Final: {filename_final}")

    # Process
    df['clean'] = df['text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', str(x).lower()))
    vec = TfidfVectorizer(max_features=1000)
    X = vec.fit_transform(df['clean']).toarray()
    y = df['label']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_test)
    print(f"\nAKURASI: {accuracy_score(y_test, y_pred)*100:.2f}%")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    plt.figure(figsize=(6,5))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.show()